In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LassoCV
from sklearn.tree import DecisionTreeRegressor, plot_tree
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from feature_engineer import get_engineered_df

In [4]:
# Insert warehouse name here
warehouse_name = "OE"

DATA_PATH = "~/Lucas_Systems_Capstone_Project/data/processed/oe_detailed.parquet"




df, features, cat_cols = get_engineered_df(DATA_PATH, warehouse="OE", max_time=300, work_code="30")

print(f"Number of rows in feature engineered data: {df.shape[0]}")


df['log_Time_Delta_sec'] = np.log1p(df['Time_Delta_sec'])
df['log_Travel_Distance'] = np.log1p(df['Travel_Distance'])
df['log_Weight'] = np.log1p(df['Weight'])
df['log_Cube'] = np.log1p(df['Cube'])
df['log_Quantity'] = np.log1p(df['Quantity'])

# List of columns to drop (IDs, raw metadata, and intermediate statistics)
cols_to_drop = [
    'ActivityCode', 'UserID', 'WorkCode', 'AssignmentID', 'ProductID', 
    'Timestamp', 'LocationID', 'Prev_Timestamp', 'Prev_LocationID', 
    'ProductCode', 'UnitOfMeasure', 'Aisle', 'Bay', 'Level', 'Slot', 
    'Prev_Aisle', 'Prev_Bay', 'Prev_Level', 'Prev_Slot', 'Aisle2', 'Bay2', 
    'Prev_Aisle2', 'Prev_Bay2', 'LocKey', 'PrevLocKey', 'hour', 'mean', 'count'
]

# Drop them while keeping Time_Delta_sec and log_Time_Delta_sec
df = df.drop(columns=cols_to_drop, errors='ignore')

# One-hot encode the categorical features created by the script (like time_of_day)
df_model = pd.get_dummies(df, columns=cat_cols, drop_first=True)


df_model = df_model.dropna()
print(f"Number of rows in feature engineered data after removing all missing values: {df_model.shape[0]}")

Number of rows in feature engineered data: 65298
Number of rows in feature engineered data after removing all missing values: 65251


# Simple OLS

In [5]:
# --- Independent Data Prep for OLS ---
# Select only log-transformed features and dummy variables
ols_features = [c for c in df_model.columns if c.startswith('log_') or any(cat + "_" in c for cat in cat_cols)]
ols_features = [c for c in ols_features if c != 'log_Time_Delta_sec']

X = df_model[ols_features].astype(float)
y = df_model["log_Time_Delta_sec"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

# Fit and Summarize
X_train_const = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_const).fit()
print(ols_model.summary())

# Evaluation (Back-transform to seconds)
pred_log = ols_model.predict(sm.add_constant(X_test))
mae_regular = mean_absolute_error(np.expm1(y_test), np.expm1(pred_log))

print(f"\n--- OLS Final Metrics ---")
print(f"Regular MAE (Seconds):    {mae_regular:.2f}s")
print(f"Test R^2:                 {r2_score(y_test, pred_log):.4f}")

                            OLS Regression Results                            
Dep. Variable:     log_Time_Delta_sec   R-squared:                       0.425
Model:                            OLS   Adj. R-squared:                  0.425
Method:                 Least Squares   F-statistic:                     1329.
Date:                Tue, 10 Mar 2026   Prob (F-statistic):               0.00
Time:                        23:03:45   Log-Likelihood:                -62330.
No. Observations:               52200   AIC:                         1.247e+05
Df Residuals:                   52170   BIC:                         1.250e+05
Df Model:                          29                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               

# Lasso

In [6]:
# --- Independent Data Prep for Lasso ---
lasso_features = [c for c in df_model.columns if c.startswith('log_') or any(cat + "_" in c for cat in cat_cols)]
lasso_features = [c for c in lasso_features if c != 'log_Time_Delta_sec']

X = df_model[lasso_features].astype(float)
y = df_model["log_Time_Delta_sec"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

# Scaling and Cross-Validation
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lasso_cv = LassoCV(cv=5, random_state=4).fit(X_train_scaled, y_train)

# Evaluation
pred_log = lasso_cv.predict(X_test_scaled)
print(f"--- Lasso CV Evaluation ---")
print(f"Best Alpha Found:         {lasso_cv.alpha_:.6f}")
print(f"Regular MAE (Seconds):    {mean_absolute_error(np.expm1(y_test), np.expm1(pred_log)):.2f}s")

# Coefficient Summary
coef_table = pd.DataFrame({"Variable": lasso_features, "Coef": lasso_cv.coef_}).sort_values(by="Coef", ascending=False)
print("\n--- Top Lasso Coefficients ---")
print(coef_table.head(10))

--- Lasso CV Evaluation ---
Best Alpha Found:         0.001166
Regular MAE (Seconds):    22.96s

--- Top Lasso Coefficients ---
               Variable      Coef
0   log_Travel_Distance  0.554129
18         diff_level_1  0.488260
3          log_Quantity  0.256606
1            log_Weight  0.088222
5        Aisle_group_41  0.020827
14     time_of_day_8-12  0.013276
4        Aisle_group_40  0.007877
22         UOM_group_PK  0.007286
10        Level_group_4  0.004592
19         UOM_group_CA -0.000000


# Decision Tree

In [7]:
# --- Independent Data Prep for Decision Tree ---
# Use regular physical features and dummy variables
tree_features = [c for c in df_model.columns if not c.startswith('log_') and c not in ['Time_Delta_sec']]

X = df_model[tree_features].astype(float)
y = df_model["Time_Delta_sec"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

# Grid Search
param_grid = {'max_depth': [None, 5, 10, 20], 'min_samples_leaf': [1, 2, 5]}
grid_dt = GridSearchCV(DecisionTreeRegressor(random_state=4), param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_dt.fit(X_train, y_train)

best_dt = grid_dt.best_estimator_
preds = best_dt.predict(X_test)

print(f"--- Decision Tree Evaluation ---")
print(f"Best Parameters:          {grid_dt.best_params_}")
print(f"Regular MAE (Seconds):    {mean_absolute_error(y_test, preds):.2f}s")
print(f"Test R^2:                 {r2_score(y_test, preds):.4f}")

# Importance
importance = pd.DataFrame({"Feature": tree_features, "Importance": best_dt.feature_importances_}).sort_values(by="Importance", ascending=False)
print("\n--- Top 10 Features ---")
print(importance.head(10))

--- Decision Tree Evaluation ---
Best Parameters:          {'max_depth': 10, 'min_samples_leaf': 5}
Regular MAE (Seconds):    23.17s
Test R^2:                 0.3530

--- Top 10 Features ---
              Feature  Importance
3     Travel_Distance    0.524046
0            Quantity    0.124841
17      same_lockey_1    0.065141
16       same_aisle_1    0.058634
2                Cube    0.056563
18       diff_level_1    0.049766
1              Weight    0.044494
7   Aisle_group_other    0.028643
6      Aisle_group_42    0.007343
4      Aisle_group_40    0.006856


# XG Boost

In [8]:
# --- Independent Data Prep for XGBoost ---
xgb_features = [c for c in df_model.columns if not c.startswith('log_') and c not in ['Time_Delta_sec']]
X = df_model[xgb_features].astype(float)
y = df_model["Time_Delta_sec"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, objective="reg:gamma", tree_method="hist", n_jobs=-1, random_state=4)
xgb_model.fit(X_train, y_train)

preds = xgb_model.predict(X_test)

print(f"--- XGBoost Model Evaluation ---")
print(f"Regular MAE (Seconds):    {mean_absolute_error(y_test, preds):.2f}s")
print(f"Test R^2:                 {r2_score(y_test, preds):.4f}")

# Importance
importance_xgb = pd.DataFrame({"Feature": xgb_features, "Importance": xgb_model.feature_importances_}).sort_values(by="Importance", ascending=False)
print("\n--- Top 10 XGBoost Features ---")
print(importance_xgb.head(10))

--- XGBoost Model Evaluation ---
Regular MAE (Seconds):    21.72s
Test R^2:                 0.3991

--- Top 10 XGBoost Features ---
              Feature  Importance
17      same_lockey_1    0.414955
18       diff_level_1    0.133778
3     Travel_Distance    0.073378
16       same_aisle_1    0.050200
7   Aisle_group_other    0.029996
27  top_100_product_1    0.028701
0            Quantity    0.026368
21       UOM_group_EA    0.018899
22       UOM_group_PK    0.018387
2                Cube    0.016049


# kNN

In [9]:
# --- Independent Data Prep for kNN ---
knn_features = [c for c in df_model.columns if not c.startswith('log_') and c not in ['Time_Delta_sec']]
X = df_model[knn_features].astype(float)
y = df_model["Time_Delta_sec"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

scaler = StandardScaler()
X_train_k = scaler.fit_transform(X_train)
X_test_k = scaler.transform(X_test)

param_grid_knn = {'n_neighbors': [5, 11, 21], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}
grid_knn = GridSearchCV(KNeighborsRegressor(), param_grid_knn, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_knn.fit(X_train_k, y_train)

print(f"--- kNN (CV Optimized) Evaluation ---")
print(f"Best Parameters:          {grid_knn.best_params_}")
print(f"Regular MAE (Seconds):    {mean_absolute_error(y_test, grid_knn.predict(X_test_k)):.2f}s")

--- kNN (CV Optimized) Evaluation ---
Best Parameters:          {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'uniform'}
Regular MAE (Seconds):    24.47s


# Neural Network

In [10]:
# --- Independent Data Prep for Neural Net ---
nn_features = [c for c in df_model.columns if not c.startswith('log_') and c not in ['Time_Delta_sec']]
X = df_model[nn_features].astype(float).values
y = df_model["Time_Delta_sec"].astype(float).values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=4)

scaler = StandardScaler()
X_train_nn = scaler.fit_transform(X_train)
X_test_nn = scaler.transform(X_test)

# Architecture
model = nn.Sequential(
    nn.Linear(X_train_nn.shape[1], 500), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(500, 200), nn.Dropout(0.2), nn.ReLU(),
    nn.Linear(200, 25), nn.ReLU(),
    nn.Linear(25, 1)
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.MSELoss()

# Loaders
train_loader = DataLoader(TensorDataset(torch.tensor(X_train_nn, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test_nn, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)), batch_size=32, shuffle=False)

# Loop
for epoch in range(10):
    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(inputs), labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluation
model.eval()
test_mae = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        test_mae += nn.L1Loss()(model(inputs), labels).item()

print(f"\n--- Neural Net Final MAE: {test_mae/len(test_loader):.2f}s ---")

Epoch 1, Loss: 1108.3539
Epoch 2, Loss: 1378.6719
Epoch 3, Loss: 2572.4158
Epoch 4, Loss: 1631.1486
Epoch 5, Loss: 1270.1981
Epoch 6, Loss: 803.3514
Epoch 7, Loss: 1529.2473
Epoch 8, Loss: 1505.3063
Epoch 9, Loss: 953.7017
Epoch 10, Loss: 973.3301

--- Neural Net Final MAE: 23.94s ---


# Random Forest

In [15]:
# 1. Setup Data - Fix for potential non-numeric types
X_complex = df_model.drop(columns=['Time_Delta_sec', 'log_Time_Delta_sec'], errors='ignore')
X_complex = X_complex.select_dtypes(include=[np.number, 'bool']).astype(float)
y_complex = df_model["Time_Delta_sec"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X_complex, y_complex, test_size=0.20, random_state=4)

# 2. Define Parameter Grid for CV
param_grid = {
    'n_estimators': [200],
    'max_depth': [15, 20],
    'min_samples_leaf': [4, 6],
    'max_features': ['sqrt', 'log2'] 
}

# 3. Random Forest with 5-Fold CV
rf_comp = RandomForestRegressor(random_state=4, n_jobs=-1)
grid_rf_complex = GridSearchCV(rf_comp, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_rf_complex.fit(X_train, y_train)

# 4. Evaluation
best_rf_complex = grid_rf_complex.best_estimator_
complex_preds = best_rf_complex.predict(X_test)

print(f"--- Complex Random Forest Results ---")
print(f"MAE (Mean Absolute Error): {mean_absolute_error(y_test, complex_preds):.2f} seconds")
print(f"R^2 Score (Goodness of Fit): {r2_score(y_test, complex_preds):.4f}")

# 5. Feature Importance (Top 10)
importances_complex = pd.Series(best_rf_complex.feature_importances_, index=X_complex.columns).sort_values(ascending=False)
print("\nTop 10 Feature Importance:")
print(importances_complex.head(10))

--- Complex Random Forest Results ---
MAE (Mean Absolute Error): 21.53 seconds
R^2 Score (Goodness of Fit): 0.4370

Top 10 Feature Importance:
Travel_Distance    0.222385
same_lockey_1      0.083050
Quantity           0.080818
same_aisle_1       0.075735
Prev_LocationID    0.048988
Prev_Bay           0.044361
mean               0.043073
Bay                0.041277
Cube               0.040709
Weight             0.039219
dtype: float64


In [16]:
summary_data_complex = {    
    "Model": ["OLS", "Lasso", "Decision Tree", "kNN", "Neural Net", "XGBoost", "Random Forest"],
    "MAE (Seconds)": [22.97, 22.96, 23.17, 24.47, 23.94, 21.72, 21.53],
    "When to Use": [
        "Baseline model, for speed and interpretability",
        "For automated feature selection for raw inputs",
        "Simple understanding of how variables split data",
        "When tasks are localized and cluster by distance",
        "For very large datasets for high accuracy",
        "For accuracy, when errors must be minimized",
        "Avoid overfitting with simple physical features"
    ],
    "When Not to Use": [
        "When non-linear patterns exist",
        "When you have very few features to penalize",
        "When precision is important (high variance)",
        "On massive datasets, computationally expensive",
        "On small datasets, when you want interpretability",
        "When compute resources are limited",
        "When model size/latency is an issue"
    ]
}

df_summary_complex = pd.DataFrame(summary_data_complex).sort_values(by="MAE (Seconds)")
display(df_summary_complex)

,Model,MAE (Seconds),When to Use,When Not to Use
6,Random Forest,21.53,Avoid overfitting with simple physical features,When model size/latency is an issue
5,XGBoost,21.72,"For accuracy, when errors must be minimized",When compute resources are limited
1,Lasso,22.96,For automated feature selection for raw inputs,When you have very few features to penalize
0,OLS,22.97,"Baseline model, for speed and interpretability",When non-linear patterns exist
2,Decision Tree,23.17,Simple understanding of how variables split data,When precision is important (high variance)
4,Neural Net,23.94,For very large datasets for high accuracy,"On small datasets, when you want interpretability"
3,kNN,24.47,When tasks are localized and cluster by distance,"On massive datasets, computationally expensive"
